In [1]:
import sys
sys.path.append("../")
from src.get_data import get_data
import pandas as pd
from  sklearn.preprocessing import StandardScaler
from src.model import get_predictions
from sklearn.metrics import classification_report
from src.generate_report import generate_report
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor


sc = StandardScaler()
lb = LabelEncoder()

df = get_data()
# generate_report(df)


/Users/rithikaramasamy/Documents/projects/MACHINE-LEARNING/ML_logistic_regression/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# EDA
# Removing boat column, because it is present only for survivors
# since we have pclass to determine their deck | cabin relation, 
# and there are too many null values in the cabin, it's better to remove the cabin data all together
df.drop(columns=["boat","body", "cabin", "home.dest", "name", "ticket","fare"], inplace=True)
# df

# before removing any column, we need to fin collinearity
# df["sex"] = lb.fit_transform(df["sex"])
# df["embarked"] = lb.fit_transform(df["embarked"])
# df

#checking null values in the data
# df.isnull().sum()



# x = df[col_list]
# vif_data = pd.DataFrame()
# vif_data["Columns"]  =  x.columns

In [3]:
#for age we can fill with median
age_med = df["age"].median()

df["age"] = df["age"].fillna(age_med)

#we can replace the fare with mode
# fare_mod = df["fare"].mode()[0]
# df["fare"] = df["fare"].fillna(df["fare"].mode()[0])

#we can replace the embarked with mode
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

# df.isnull().sum()
import numpy as np



#VIF value:
col_list =[]
for col in df.columns:
    if((df[col].dtype != "object") & (col != "survived")):
        col_list.append(col)
        
# X = df[col_list].copy()

# print("NaNs per column:\n", X.isna().sum().sort_values(ascending=False).head(20))
# print("Any infs?", np.isinf(X.to_numpy()).any())
X = df[col_list]
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values,i)
                   for i in range(len(X.columns))]
print(vif_data)



  feature       VIF
0  pclass  3.196281
1     age  2.898575
2   sibsp  1.431197
3   parch  1.380034


In [4]:

#need to convert the cat value to num values:
df = pd.get_dummies(df, columns=["embarked", "sex",], drop_first=True)
df 

,pclass,survived,age,sibsp,parch,embarked_Q,embarked_S,sex_male
0,1,1,29.0000,0,0,False,True,False
1,1,1,0.9167,1,2,False,True,True
2,1,0,2.0000,1,2,False,True,False
3,1,0,30.0000,1,2,False,True,True
4,1,0,25.0000,1,2,False,True,False
...,...,...,...,...,...,...,...,...
1304,3,0,14.5000,1,0,False,False,False
1305,3,0,28.0000,1,0,False,False,False
1306,3,0,26.5000,0,0,False,False,True
1307,3,0,27.0000,0,0,False,False,True


In [5]:
df_independant = df.drop(columns=["survived"])
df_target = df["survived"]

predictions, y_test, probs, X_train , X_test, y_train = get_predictions(independent=df_independant, target=df_target)

# predictions, y_test
print(classification_report(predictions, y_test))



              precision    recall  f1-score   support

           0       0.81      0.81      0.81       144
           1       0.77      0.77      0.77       118

    accuracy                           0.79       262
   macro avg       0.79      0.79      0.79       262
weighted avg       0.79      0.79      0.79       262

